In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.pipeline import Pipeline

In [ ]:
df = pd.read_csv('/content/sample_data/train.csv')

In [ ]:
df.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
# X = df[['Neighborhood', 'LotArea', 'PoolArea']]
# y = df['SalePrice']

In [ ]:
test = pd.read_csv('/content/sample_data/test.csv')

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42)

In [ ]:
from sklearn.compose import make_column_transformer
# transformer = make_column_transformer((OneHotEncoder(drop = 'first', handle_unknown = 'ignore'), ['Neighborhood']),
#                                       remainder = 'passthrough')

# pipe = Pipeline([('transform', transformer), ('model', LinearRegression())])

# pipe.fit(X_train, y_train)
# preds = pipe.predict(X_test)

In [ ]:
# mean_squared_error(y_test, preds) ** 0.5

In [ ]:
df.select_dtypes(['int', 'float']).corr()['SalePrice'].sort_values(ascending = False)

,SalePrice
SalePrice,1.000000
OverallQual,0.789142
GrLivArea,0.740379
TotalBsmtSF,0.641917
GarageCars,0.640961
GarageArea,0.623329
1stFlrSF,0.618556
FullBath,0.571867
TotRmsAbvGrd,0.559435
YearBuilt,0.523194


In [ ]:
#get all the columns with missing values
df.columns[df.isna().sum() > 0]

#percentage of missing values
df[df.columns[df.isna().sum() > 0]].isna().sum()/len(df)

,0
LotFrontage,0.175000
Alley,0.937500
MasVnrType,0.597500
MasVnrArea,0.005000
BsmtQual,0.026667
BsmtCond,0.026667
BsmtExposure,0.027500
BsmtFinType1,0.026667
BsmtFinType2,0.027500
FireplaceQu,0.470000


In [ ]:
df = df.drop(columns = ['PoolQC' , 'Fence', 'MiscFeature', 'FireplaceQu', 'Alley'])

In [ ]:
# numeric columns
num_cols = ['OverallQual', 'GrLivArea', 'TotalBsmtSF', '1stFlrSF', 'GarageCars',
            'GarageArea', 'FullBath', 'TotRmsAbvGrd', 'Fireplaces',
            'YearBuilt', 'YearRemodAdd', 'LotArea', 'MasVnrArea']

# quality columns that have an order
ord_cols = ['ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC', 'GarageQual']

# categories with no order
ohe_cols = ['Neighborhood', 'MSZoning', 'BldgType', 'Foundation',
            'GarageType', 'CentralAir', 'SaleCondition']

cols = num_cols + ord_cols + ohe_cols

In [ ]:
vals = [['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']]

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
ohe_pipe = Pipeline([('impute', SimpleImputer(strategy = 'constant', fill_value = 'None')),
                     ('ohe', OneHotEncoder(drop = 'first', handle_unknown = 'ignore'))])


qual = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
ord_vals = [qual, qual, qual, qual, qual, qual,
            ['None', 'No', 'Mn', 'Av', 'Gd'],    # BsmtExposure
            ['None', 'Unf', 'RFn', 'Fin']]       # GarageFinish

ord_pipe = Pipeline([('impute', SimpleImputer(strategy = 'constant', fill_value = 'None')),
                     ('ord', OrdinalEncoder(categories = vals*len(ord_cols)))])

transformer = make_column_transformer((ohe_pipe, ohe_cols),
                                      (ord_pipe, ord_cols),
                                      (SimpleImputer(), num_cols))

In [ ]:
X = df[cols]
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42)

In [ ]:
model_pipeline = Pipeline([('transform', transformer), ('model', LinearRegression())])

model_pipeline.fit(X, y)
preds = model_pipeline.predict(X_test)

mean_squared_error(y_test, preds) ** 0.5

33570.42799441155

In [ ]:
pdf = pd.DataFrame({'Id': test.index, 'SalePrice': test_preds})
pdf.to_csv('preds.csv', index = False)